# LIMPIEZA DE DATOS - ÍNDICE DE PRODUCCIÓN INDUSTRIAL (IPI)
### Base 2021 del INE
### Incluye:
- Importación de librerías
- Carga de datasets: IPI (volumen), IPRI (precios industriales)
- Filtrado: Total Nacional, solo índices (sin tasas de variación)
- Limpieza y pivotado por destino económico de los bienes
- **Eliminación de sub-componentes de consumo** (duradero/no duradero) por ruptura de chain-linking
- Conversión a precios corrientes: IPI × IPRI / 100
- Deflactación a precios constantes base 2025: corrientes / (IPRI₂₀₂₅ / 100)
- Visualización de las series
- Exportación de archivos finales

---
**Fuentes:**
- `Índice de Producción Industrial. (Base 2021).csv` - INE. Cobertura: enero 1975 , marzo 2026. Mensual. Índice de volumen (base 2021 = 100).
- `indice_precios_industriales_IPRI_bases_2021_2025.csv` - NB5. Cobertura: enero 1975 , marzo 2026. Mensual. Índice de precios industriales en bases 2021 y 2025.

# 0. Importación de librerías

In [24]:
import pandas as pd
import numpy as np
import plotly.express as px

# 1. Constantes globales

In [25]:

ARCHIVO_IPI  = '../Fuentes/INE/Índice de Producción Industrial/Índice de Producción Industrial, (Base 2021).csv'
ARCHIVO_IPRI = '../Datasets/indice_precios_industriales_IPRI_bases_2021_2025.csv'
ARCHIVO_IPC  = '../Datasets/indice_precios_consumo_IPC_diferentes_bases.csv'

# Categorías IPI por destino económico de los bienes (Total Nacional)

CATEGORIAS_IPI = [
    'Total industria',
    'Energía',
    'Bienes de equipo',
    'Bienes intermedios',
    'Bienes de consumo',
]

# Categorías con anomalía de encadenamiento (se cargan para diagnóstico, se eliminan antes del cálculo)
CATEGORIAS_IPI_DIAGNOSTICO = [
    'Bienes de consumo duradero',
    'Bienes de consumo no duradero',
]

# Nombres internos para las columnas del IPI (volumen)
NOMBRES_IPI = {
    'Total industria':              'ipi_total_industria',
    'Energía':                      'ipi_energia',
    'Bienes de equipo':             'ipi_bienes_equipo',
    'Bienes intermedios':           'ipi_bienes_intermedios',
    'Bienes de consumo':            'ipi_bienes_consumo',
}

NOMBRES_IPI_DIAGNOSTICO = {
    'Bienes de consumo duradero':   'ipi_bienes_consumo_duradero',
    'Bienes de consumo no duradero':'ipi_bienes_consumo_no_duradero',
}

# Nombres para exportación: prefijo distingue corrientes vs constantes
NOMBRES_CORRIENTES = {v: v.replace('ipi_', 'ipi_corrientes_')
                      for v in NOMBRES_IPI.values()}
NOMBRES_CONSTANTES = {v: v.replace('ipi_', 'ipi_constantes_2025_')
                      for v in NOMBRES_IPI.values()}

# Correspondencia IPI → IPRI (base 2021) para el cálculo de precios corrientes
IPI_A_IPRI_2021 = {
    'ipi_total_industria':          'IPRI_total_industria_2021',
    'ipi_energia':                  'IPRI_energia_2021',
    'ipi_bienes_equipo':            'IPRI_bienes_equipo_2021',
    'ipi_bienes_intermedios':       'IPRI_bienes_intermedios_2021',
    'ipi_bienes_consumo':           'IPRI_bienes_consumo_2021',
}

# Correspondencia IPI → IPRI (base 2025) para la deflactación
IPI_A_IPRI_2025 = {
    'ipi_total_industria':          'IPRI_total_industria_2025',
    'ipi_energia':                  'IPRI_energia_2025',
    'ipi_bienes_equipo':            'IPRI_bienes_equipo_2025',
    'ipi_bienes_intermedios':       'IPRI_bienes_intermedios_2025',
    'ipi_bienes_consumo':           'IPRI_bienes_consumo_2025',
}

print('Categorías IPI (finales):', len(CATEGORIAS_IPI))
print('Categorías diagnóstico (se eliminarán):', len(CATEGORIAS_IPI_DIAGNOSTICO))
print('Correspondencias IPI→IPRI definidas:', len(IPI_A_IPRI_2021))

Categorías IPI (finales): 5
Categorías diagnóstico (se eliminarán): 2
Correspondencias IPI→IPRI definidas: 5


# 2. Carga de datasets

### 2.1 Índice de Producción Industrial (IPI)
Archivo del INE con formato: `Comunidades y Ciudades Autónomas; Destino económico de los bienes; Índice y tasas; Periodo; Total`.

Filtros aplicados:
- **Ámbito:** `Total Nacional` (se descartan comunidades autónomas)
- **Tipo:** `Índice` (se descartan tasas de variación anual, mensual, etc.)

In [26]:
# Carga del IPI
df_ipi_raw = pd.read_csv(ARCHIVO_IPI, sep=';', encoding='utf-8-sig')
print(f'IPI bruto: {df_ipi_raw.shape}')
print(f'Columnas: {df_ipi_raw.columns.tolist()}')
print()

# Valores únicos por campo
for col in df_ipi_raw.columns[:-1]:
    vals = df_ipi_raw[col].dropna().unique()
    print(f'{col}: {len(vals)} valores únicos')
    if len(vals) <= 10:
        for v in sorted(vals):
            print(f'  • {v}')
    print()

IPI bruto: (309960, 5)
Columnas: ['Comunidades y Ciudades Autónomas', 'Destino económico de los bienes', 'Índice y tasas', 'Periodo', 'Total']

Comunidades y Ciudades Autónomas: 18 valores únicos

Destino económico de los bienes: 7 valores únicos
  • Bienes de consumo
  • Bienes de consumo duradero
  • Bienes de consumo no duradero
  • Bienes de equipo
  • Bienes intermedios
  • Energía
  • Total industria

Índice y tasas: 4 valores únicos
  • Variación anual
  • Variación de la media en lo que va de año
  • Variación mensual
  • Índice

Periodo: 615 valores únicos



In [27]:
# Filtrado: Total Nacional + solo Índice + TODAS las categorías (incluidas las de diagnóstico)
todas_categorias = CATEGORIAS_IPI + CATEGORIAS_IPI_DIAGNOSTICO

filtro = (
    (df_ipi_raw['Comunidades y Ciudades Autónomas'] == 'Total Nacional') &
    (df_ipi_raw['Índice y tasas'] == 'Índice') &
    (df_ipi_raw['Destino económico de los bienes'].isin(todas_categorias))
)
df_ipi = df_ipi_raw[filtro].copy()
print(f'Tras filtrado: {df_ipi.shape}')

# Parsear periodo: formato YYYYMmm → fecha (primer día del mes)
s = df_ipi['Periodo'].astype(str).str.strip()
df_ipi['año'] = s.str[:4].astype(int)
df_ipi['mes'] = s.str[5:].astype(int)
df_ipi['fecha'] = pd.to_datetime(dict(year=df_ipi['año'], month=df_ipi['mes'], day=1))

# Parsear valor numérico (formato español: coma decimal)
df_ipi['valor'] = (
    df_ipi['Total'].astype(str).str.strip()
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)
df_ipi['valor'] = pd.to_numeric(df_ipi['valor'], errors='coerce')

# Verificar que errors='coerce' no haya generado nulos inesperados
n_coerced = df_ipi['valor'].isna().sum()
print(f'  Valores no parseables convertidos a NaN: {n_coerced}')
if n_coerced > 0:
    print(f'ALERTA  {n_coerced} valores no numéricos detectados — revisar columna "Total" del INE')

# Renombrar categorías a nombres internos
todos_nombres = {**NOMBRES_IPI, **NOMBRES_IPI_DIAGNOSTICO}
df_ipi['variable'] = df_ipi['Destino económico de los bienes'].map(todos_nombres)

print(f'Rango temporal: {df_ipi["fecha"].min().date()} → {df_ipi["fecha"].max().date()}')
print(f'Categorías: {df_ipi["variable"].unique().tolist()}')

Tras filtrado: (4305, 5)
  Valores no parseables convertidos a NaN: 0
Rango temporal: 1975-01-01 → 2026-03-01
Categorías: ['ipi_total_industria', 'ipi_bienes_consumo', 'ipi_bienes_consumo_duradero', 'ipi_bienes_consumo_no_duradero', 'ipi_bienes_equipo', 'ipi_bienes_intermedios', 'ipi_energia']


In [28]:
# Pivotear: una columna por categoría (incluye las de diagnóstico por ahora)
df_ipi_pivot = df_ipi.pivot_table(
    index=['fecha', 'año', 'mes'],
    columns='variable',
    values='valor',
    aggfunc='first'
).reset_index()
df_ipi_pivot.columns.name = None

# Ordenar columnas: primero las finales, luego las de diagnóstico
todos_nombres = {**NOMBRES_IPI, **NOMBRES_IPI_DIAGNOSTICO}
cols_todas = [todos_nombres[c] for c in CATEGORIAS_IPI + CATEGORIAS_IPI_DIAGNOSTICO]
cols_ipi = [NOMBRES_IPI[c] for c in CATEGORIAS_IPI]
cols_diag = [NOMBRES_IPI_DIAGNOSTICO[c] for c in CATEGORIAS_IPI_DIAGNOSTICO]

df_ipi_pivot = df_ipi_pivot[['fecha', 'año', 'mes'] + cols_todas].sort_values('fecha').reset_index(drop=True)

print(f'IPI pivotado: {df_ipi_pivot.shape} (incluye {len(cols_diag)} columnas de diagnóstico)')
print(f'Rango: {df_ipi_pivot["fecha"].min().date()} → {df_ipi_pivot["fecha"].max().date()}')
print(f'Columnas finales: {cols_ipi}')
print(f'Columnas diagnóstico: {cols_diag}')
print()
print(df_ipi_pivot.head(3).to_string(index=False))
print('...')
print(df_ipi_pivot.tail(3).to_string(index=False))

IPI pivotado: (615, 10) (incluye 2 columnas de diagnóstico)
Rango: 1975-01-01 → 2026-03-01
Columnas finales: ['ipi_total_industria', 'ipi_energia', 'ipi_bienes_equipo', 'ipi_bienes_intermedios', 'ipi_bienes_consumo']
Columnas diagnóstico: ['ipi_bienes_consumo_duradero', 'ipi_bienes_consumo_no_duradero']

     fecha  año  mes  ipi_total_industria  ipi_energia  ipi_bienes_equipo  ipi_bienes_intermedios  ipi_bienes_consumo  ipi_bienes_consumo_duradero  ipi_bienes_consumo_no_duradero
1975-01-01 1975    1               68.584      106.785             99.416                  86.662              89.638                       90.702                          89.547
1975-02-01 1975    2               70.061       47.219            127.735                  58.997              51.341                      445.467                          44.889
1975-03-01 1975    3               73.509       46.875            165.914                  58.343              50.893                      420.806           

### 2.2 Índice de Precios Industriales (IPRI)
Archivo ya procesado con columnas en bases 2021 y 2025 por categoría.

In [29]:
# Carga del IPRI
df_ipri = pd.read_csv(ARCHIVO_IPRI, encoding='utf-8-sig')
df_ipri['fecha'] = pd.to_datetime(df_ipri['fecha'])

print(f'IPRI: {df_ipri.shape}')
print(f'Rango: {df_ipri["fecha"].min().date()} → {df_ipri["fecha"].max().date()}')
print(f'Columnas: {df_ipri.columns.tolist()}')
print()

# Verificar que las columnas necesarias existen
cols_necesarias_2021 = list(IPI_A_IPRI_2021.values())
cols_necesarias_2025 = list(IPI_A_IPRI_2025.values())
faltantes = [c for c in cols_necesarias_2021 + cols_necesarias_2025 if c not in df_ipri.columns]
if faltantes:
    print(f'ALERTA  Columnas faltantes en IPRI: {faltantes}')
else:
    print('CORRECTO  Todas las columnas IPRI necesarias están presentes')

IPRI: (610, 24)
Rango: 1975-01-01 → 2025-10-01
Columnas: ['fecha', 'año', 'trimestre', 'periodo', 'IPRI_total_industria_2021', 'IPRI_industrias_extractivas_2021', 'IPRI_industria_manufacturera_2021', 'IPRI_suministro_energia_electrica_gas_vapor_aire_acondicionado_2021', 'IPRI_energia_2021', 'IPRI_bienes_equipo_2021', 'IPRI_bienes_intermedios_2021', 'IPRI_bienes_consumo_2021', 'IPRI_bienes_consumo_duradero_2021', 'IPRI_bienes_consumo_no_duradero_2021', 'IPRI_total_industria_2025', 'IPRI_industrias_extractivas_2025', 'IPRI_industria_manufacturera_2025', 'IPRI_suministro_energia_electrica_gas_vapor_aire_acondicionado_2025', 'IPRI_energia_2025', 'IPRI_bienes_equipo_2025', 'IPRI_bienes_intermedios_2025', 'IPRI_bienes_consumo_2025', 'IPRI_bienes_consumo_duradero_2025', 'IPRI_bienes_consumo_no_duradero_2025']

CORRECTO  Todas las columnas IPRI necesarias están presentes


# 3. Verificación de cobertura temporal

El IPI y el IPRI deben cubrir el mismo rango mensual sin huecos para poder multiplicarlos.

In [30]:
# Verificando la serie mensual continua para IPI
rango_ipi = pd.date_range(df_ipi_pivot['fecha'].min(), df_ipi_pivot['fecha'].max(), freq='MS')
gaps_ipi = rango_ipi.difference(pd.DatetimeIndex(df_ipi_pivot['fecha']))
print(f'IPI:')
print(f'  Rango: {df_ipi_pivot["fecha"].min().date()} → {df_ipi_pivot["fecha"].max().date()}')
print(f'  Observaciones: {len(df_ipi_pivot)} (esperadas: {len(rango_ipi)})')
if len(gaps_ipi) > 0:
    print(f'ALERTA  {len(gaps_ipi)} meses faltantes')
else:
    print(f'CORRECTO  Serie mensual continua')
print()

# Verificar cobertura IPRI
rango_ipri = pd.date_range(df_ipri['fecha'].min(), df_ipri['fecha'].max(), freq='MS')
gaps_ipri = rango_ipri.difference(pd.DatetimeIndex(df_ipri['fecha']))
print(f'IPRI:')
print(f'  Rango: {df_ipri["fecha"].min().date()} → {df_ipri["fecha"].max().date()}')
print(f'  Observaciones: {len(df_ipri)} (esperadas: {len(rango_ipri)})')
if len(gaps_ipri) > 0:
    print(f'ALERTA  {len(gaps_ipri)} meses faltantes')
else:
    print(f'CORRECTO  Serie mensual continua')
print()

# Verificar alineación entre IPI e IPRI
fechas_ipi  = set(df_ipi_pivot['fecha'])
fechas_ipri = set(df_ipri['fecha'])
solo_ipi  = fechas_ipi - fechas_ipri
solo_ipri = fechas_ipri - fechas_ipi

if len(solo_ipi) == 0 and len(solo_ipri) == 0:
    print('CORRECTO  IPI e IPRI cubren exactamente el mismo período')
else:
    if solo_ipi:
        print(f'ALERTA  {len(solo_ipi)} meses en IPI sin IPRI: {sorted(solo_ipi)[:5]}...')
    if solo_ipri:
        print(f'ALERTA  {len(solo_ipri)} meses en IPRI sin IPI: {sorted(solo_ipri)[:5]}...')

IPI:
  Rango: 1975-01-01 → 2026-03-01
  Observaciones: 615 (esperadas: 615)
CORRECTO  Serie mensual continua

IPRI:
  Rango: 1975-01-01 → 2025-10-01
  Observaciones: 610 (esperadas: 610)
CORRECTO  Serie mensual continua

ALERTA  5 meses en IPI sin IPRI: [Timestamp('2025-11-01 00:00:00'), Timestamp('2025-12-01 00:00:00'), Timestamp('2026-01-01 00:00:00'), Timestamp('2026-02-01 00:00:00'), Timestamp('2026-03-01 00:00:00')]...


# 4. Análisis de valores nulos

Se verifica que no haya huecos en las series del IPI por categoría.

In [31]:
print('=== IPI — Valores nulos por categoría ===')
for col in cols_ipi:
    n_nulos = df_ipi_pivot[col].isna().sum()
    if n_nulos > 0:
        primer = df_ipi_pivot[df_ipi_pivot[col].notna()]['fecha'].min().date()
        print(f'ALERTA  {col}: {n_nulos} nulos (disponible desde {primer})')
    else:
        print(f'CORRECTO  {col}: sin nulos ({len(df_ipi_pivot)} observaciones)')

print()
print('=== IPRI — Valores nulos en columnas usadas ===')
for col_ipri in list(IPI_A_IPRI_2021.values()) + list(IPI_A_IPRI_2025.values()):
    n_nulos = df_ipri[col_ipri].isna().sum()
    if n_nulos > 0:
        print(f'ALERTA  {col_ipri}: {n_nulos} nulos')
    else:
        print(f'CORRECTO  {col_ipri}: sin nulos')

=== IPI — Valores nulos por categoría ===
CORRECTO  ipi_total_industria: sin nulos (615 observaciones)
CORRECTO  ipi_energia: sin nulos (615 observaciones)
CORRECTO  ipi_bienes_equipo: sin nulos (615 observaciones)
CORRECTO  ipi_bienes_intermedios: sin nulos (615 observaciones)
CORRECTO  ipi_bienes_consumo: sin nulos (615 observaciones)

=== IPRI — Valores nulos en columnas usadas ===
CORRECTO  IPRI_total_industria_2021: sin nulos
CORRECTO  IPRI_energia_2021: sin nulos
CORRECTO  IPRI_bienes_equipo_2021: sin nulos
CORRECTO  IPRI_bienes_intermedios_2021: sin nulos
CORRECTO  IPRI_bienes_consumo_2021: sin nulos
CORRECTO  IPRI_total_industria_2025: sin nulos
CORRECTO  IPRI_energia_2025: sin nulos
CORRECTO  IPRI_bienes_equipo_2025: sin nulos
CORRECTO  IPRI_bienes_intermedios_2025: sin nulos
CORRECTO  IPRI_bienes_consumo_2025: sin nulos


# 5. Verificación de consistencia

Se verifica que en el año base 2021, todas las categorías promedien ~100.

In [32]:
# Verificar que en base 2021, los valores del año 2021 promedian ~100
print('Verificación — media año 2021 (base 2021 = 100):')
for col in cols_ipi:
    media_2021 = df_ipi_pivot.loc[df_ipi_pivot['año'] == 2021, col].mean()
    ok = 'CORRECTO' if 99.0 <= media_2021 <= 101.0 else 'ALERTA'
    print(f'  {ok} {col}: {media_2021:.2f}')

Verificación — media año 2021 (base 2021 = 100):
  CORRECTO ipi_total_industria: 100.00
  CORRECTO ipi_energia: 100.00
  CORRECTO ipi_bienes_equipo: 100.00
  CORRECTO ipi_bienes_intermedios: 100.00
  CORRECTO ipi_bienes_consumo: 100.00


# 6. Conversión a precios corrientes

El IPI es un índice de **volumen** (base 2021 = 100): mide cambios en la producción física.  
Para obtener un índice en **precios corrientes** (nominal), se multiplica cada categoría del IPI por su IPRI homólogo:

$$\text{IPI\_corrientes}_i = \frac{\text{IPI\_volumen}_i \times \text{IPRI}_i^{2021}}{100}$$

El resultado es un índice de valor que refleja tanto los cambios en volumen como en precios.

In [33]:
# Merge IPI con IPRI por fecha
cols_ipri_necesarias = (
    ['fecha']
    + list(IPI_A_IPRI_2021.values())
    + list(IPI_A_IPRI_2025.values())
)
df_merge = df_ipi_pivot.merge(
    df_ipri[cols_ipri_necesarias],
    on='fecha',
    how='inner'
)
print(f'Merge IPI × IPRI: {df_merge.shape[0]} observaciones')
print(f'Rango: {df_merge["fecha"].min().date()} → {df_merge["fecha"].max().date()}')

# Calcular IPI en precios corrientes: IPI_vol × IPRI_2021 / 100
df_corrientes = df_merge[['fecha', 'año', 'mes']].copy()

for col_ipi, col_ipri in IPI_A_IPRI_2021.items():
    df_corrientes[col_ipi] = df_merge[col_ipi] * df_merge[col_ipri] / 100

print()
print('IPI en precios corrientes (primeras 3 filas):')
print(df_corrientes.head(3).to_string(index=False))
print('...')
print(df_corrientes.tail(3).to_string(index=False))

Merge IPI × IPRI: 610 observaciones
Rango: 1975-01-01 → 2025-10-01

IPI en precios corrientes (primeras 3 filas):
     fecha  año  mes  ipi_total_industria  ipi_energia  ipi_bienes_equipo  ipi_bienes_intermedios  ipi_bienes_consumo
1975-01-01 1975    1             7.407072     6.449814          11.800679               11.266060           10.792415
1975-02-01 1975    2             7.713716     3.225058          15.583670                7.752206            6.222529
1975-03-01 1975    3             8.144797     3.229687          20.523562                7.666270            6.208946
...
     fecha  año  mes  ipi_total_industria  ipi_energia  ipi_bienes_equipo  ipi_bienes_intermedios  ipi_bienes_consumo
2025-08-01 2025    8            98.500836   142.881159          74.789021               75.752209          103.820332
2025-09-01 2025    9           133.931734   132.796503         134.220327              113.443043          139.048325
2025-10-01 2025   10           140.487339   135.191272  

In [34]:
# Verificación: en 2021, IPI_vol ≈ 100 y IPRI_2021 ≈ 100,
# por tanto IPI_corrientes ≈ 100 en 2021
print('Verificación — media 2021 de IPI corrientes (debería ≈ 100):')
for col in cols_ipi:
    media = df_corrientes.loc[df_corrientes['año'] == 2021, col].mean()
    print(f'  {col}: {media:.2f}')

Verificación — media 2021 de IPI corrientes (debería ≈ 100):
  ipi_total_industria: 100.08
  ipi_energia: 100.54
  ipi_bienes_equipo: 99.99
  ipi_bienes_intermedios: 99.93
  ipi_bienes_consumo: 100.05


# 7. Deflactación a precios constantes base 2025

Para obtener el IPI en **precios constantes base 2025**, se deflacta el índice en precios corrientes por el IPRI en base 2025:

$$\text{IPI\_constantes\_2025}_i = \frac{\text{IPI\_corrientes}_i}{\text{IPRI}_i^{2025} / 100}$$

Esto elimina el efecto precio y deja solo el componente de volumen, expresado en precios de 2025 (donde IPRI₂₀₂₅ promedia 100 en 2025).

In [35]:
# Calcular IPI en precios constantes base 2025
# Reutiliza df_corrientes (ya calculado) en lugar de recomputar desde df_merge
df_constantes = df_merge[['fecha', 'año', 'mes']].copy()

for col_ipi, col_ipri_2025 in IPI_A_IPRI_2025.items():
    df_constantes[col_ipi] = df_corrientes[col_ipi] / (df_merge[col_ipri_2025] / 100)

print(f'IPI constantes base 2025: {df_constantes.shape}')
print()
print('Primeras 3 filas:')
print(df_constantes.head(3).to_string(index=False))
print('...')
print(df_constantes.tail(3).to_string(index=False))

IPI constantes base 2025: (610, 8)

Primeras 3 filas:
     fecha  año  mes  ipi_total_industria  ipi_energia  ipi_bienes_equipo  ipi_bienes_intermedios  ipi_bienes_consumo
1975-01-01 1975    1            86.051773   142.070324         112.418784               98.775903          111.397625
1975-02-01 1975    2            87.904953    62.821732         144.441674               67.243797           63.804028
1975-03-01 1975    3            92.231130    62.364063         187.614169               66.498379           63.247276
...
     fecha  año  mes  ipi_total_industria  ipi_energia  ipi_bienes_equipo  ipi_bienes_intermedios  ipi_bienes_consumo
2025-08-01 2025    8            98.633821   142.863262          74.795084               76.131828          103.799451
2025-09-01 2025    9           134.790271   135.478026         133.970543              114.202871          138.452292
2025-10-01 2025   10           140.632115   135.490000         142.907189              120.514991          143.58236

In [36]:
# Verificación: en 2025, los precios constantes deberían ≈ corrientes
# (porque IPRI_2025 ≈ 100 en 2025)
print('Verificación — ratio constantes/corrientes en 2025 (debería ≈ 1.0):')
mask_2025 = df_corrientes['año'] == 2025
for col in cols_ipi:
    ratio = df_constantes.loc[mask_2025, col].mean() / df_corrientes.loc[mask_2025, col].mean()
    print(f'  {col}: {ratio:.4f}')

print()
print('Verificación — ratio constantes/corrientes en 1975 (efecto acumulado de precios):')
mask_1975 = df_corrientes['año'] == 1975
for col in cols_ipi:
    ratio = df_constantes.loc[mask_1975, col].mean() / df_corrientes.loc[mask_1975, col].mean()
    print(f'  {col}: {ratio:.4f}')

Verificación — ratio constantes/corrientes en 2025 (debería ≈ 1.0):
  ipi_total_industria: 0.9994
  ipi_energia: 0.9945
  ipi_bienes_equipo: 1.0011
  ipi_bienes_intermedios: 0.9988
  ipi_bienes_consumo: 1.0011

Verificación — ratio constantes/corrientes en 1975 (efecto acumulado de precios):
  ipi_total_industria: 11.1696
  ipi_energia: 19.5006
  ipi_bienes_equipo: 8.8709
  ipi_bienes_intermedios: 8.6560
  ipi_bienes_consumo: 9.9986


# 8. Visualización de las series

In [37]:
# IPI volumen (base 2021): TODAS las categorías (incluye anomalía en duradero)
fig_vol_raw = px.line(
    df_ipi_pivot.melt('fecha', value_vars=cols_todas),
    x='fecha', y='value', color='variable',
    title='IPI Volumen (base 2021) — ANTES de limpieza: anomalía visible en bienes_consumo_duradero',
    labels={'value': 'Índice', 'fecha': '', 'variable': 'Categoría'}
)
fig_vol_raw.show()

### 8.1 Eliminación de sub-componentes de bienes de consumo debido a ruptura de encadenamiento

**Problema:** `ipi_bienes_consumo_duradero` presenta valores de 400–900 en el período 1975–1991, con una caída abrupta del 53% en un solo mes (De Septiembre a Octubre de 1991: de 597.9 a 280.2). Todas las demás categorías muestran transiciones suaves (ratios 1991/1992 entre 1.00 y 1.06).

**Causa:** Artefacto del encadenamiento retrospectivo que el INE aplica al retroextrapolar la serie base 2021 hacia períodos anteriores. Cada cambio de base del IPI (1972, 1985, 2000, 2005, 2010, 2015, 2021) implica:
- **Cambio de clasificación industrial**: la transición de CNAE-74 a CNAE-93 (1991-1993), alineada con NACE Rev.1 de la UE, redefinió qué productos pertenecen a cada categoría por destino económico.
- **Cambio de composición**: productos que en bases anteriores se clasificaban como "bienes de consumo duradero" pasaron a categorías como "bienes de equipo" o "bienes intermedios" en bases posteriores.
- **Cambio de ponderaciones**: al actualizar los pesos relativos de cada producto dentro de la categoría.

El encadenamiento multiplica tasas de crecimiento a través de estos cambios. Para categorías donde la composición cambia radicalmente, los niveles acumulados pierden significado económico aunque las tasas a corto plazo puedan ser aproximadamente correctas.

El agregado `bienes_consumo` no está afectado (rango 53.8–109.4, sin discontinuidad) porque los cambios de composición entre sub-categorías se compensan a nivel del agregado.

**Decisión:** Se eliminan ambos sub-componentes (`bienes_consumo_duradero` y `bienes_consumo_no_duradero`). Aunque `no_duradero` está limpio individualmente, conservar solo un sub-componente impide verificar la descomposición (consumo = duradero + no duradero).

In [38]:
# Eliminar sub-componentes de bienes de consumo
cols_drop = list(NOMBRES_IPI_DIAGNOSTICO.values())
df_ipi_pivot.drop(columns=cols_drop, inplace=True)
print(f'Columnas eliminadas: {cols_drop}')
print(f'df_ipi_pivot tras limpieza: {df_ipi_pivot.shape}')
print(f'Columnas restantes: {df_ipi_pivot.columns.tolist()}')

# Actualizar cols_ipi (ya definido arriba sin duradero/no duradero)
cols_ipi = [NOMBRES_IPI[c] for c in CATEGORIAS_IPI]
print(f'Categorías finales para cálculos: {cols_ipi}')
print()

# Gráfico limpio
fig_vol = px.line(
    df_ipi_pivot.melt('fecha', value_vars=cols_ipi),
    x='fecha', y='value', color='variable',
    title='IPI Volumen (base 2021) — DESPUÉS de limpieza: 5 categorías fiables',
    labels={'value': 'Índice', 'fecha': '', 'variable': 'Categoría'}
)
fig_vol.show()

Columnas eliminadas: ['ipi_bienes_consumo_duradero', 'ipi_bienes_consumo_no_duradero']
df_ipi_pivot tras limpieza: (615, 8)
Columnas restantes: ['fecha', 'año', 'mes', 'ipi_total_industria', 'ipi_energia', 'ipi_bienes_equipo', 'ipi_bienes_intermedios', 'ipi_bienes_consumo']
Categorías finales para cálculos: ['ipi_total_industria', 'ipi_energia', 'ipi_bienes_equipo', 'ipi_bienes_intermedios', 'ipi_bienes_consumo']



In [39]:
# IPI precios corrientes: total industria
fig_corr = px.line(
    df_corrientes.melt('fecha', value_vars=cols_ipi),
    x='fecha', y='value', color='variable',
    title='IPI en precios corrientes (IPI × IPRI / 100)',
    labels={'value': 'Índice corriente', 'fecha': '', 'variable': 'Categoría'}
)
fig_corr.show()

In [40]:
# IPI constantes base 2025: total industria
fig_const = px.line(
    df_constantes.melt('fecha', value_vars=cols_ipi),
    x='fecha', y='value', color='variable',
    title='IPI en precios constantes base 2025',
    labels={'value': 'Índice constante 2025', 'fecha': '', 'variable': 'Categoría'}
)
fig_const.show()

In [41]:
# Comparación corrientes vs constantes para Total industria
df_comp = pd.DataFrame({
    'fecha': df_corrientes['fecha'],
    'Corrientes': df_corrientes['ipi_total_industria'],
    'Constantes base 2025': df_constantes['ipi_total_industria'],
})

fig_comp = px.line(
    df_comp.melt('fecha'),
    x='fecha', y='value', color='variable',
    title='IPI Total industria — Corrientes vs Constantes base 2025',
    labels={'value': 'Índice', 'fecha': '', 'variable': ''}
)
fig_comp.show()

# 9. Resumen de los datasets finales

In [42]:
cols_ipi = [NOMBRES_IPI[c] for c in CATEGORIAS_IPI]

for nombre, df, mapa in [
    ('IPI corrientes',          df_corrientes, NOMBRES_CORRIENTES),
    ('IPI constantes base 2025', df_constantes, NOMBRES_CONSTANTES),
]:
    print(f'=== {nombre} ===')
    print(f'  Shape: {df.shape}')
    print(f'  Rango: {df["fecha"].min().date()} → {df["fecha"].max().date()}')
    print(f'  Frecuencia: mensual')
    print(f'  Columnas internas: {cols_ipi[:3]} ...')
    print(f'  Columnas export:   {list(mapa.values())[:3]} ...')
    print(f'  Nulos:')
    for col in cols_ipi:
        n = df[col].isna().sum()
        print(f'    {col}: {n}')
    print()

=== IPI corrientes ===
  Shape: (610, 8)
  Rango: 1975-01-01 → 2025-10-01
  Frecuencia: mensual
  Columnas internas: ['ipi_total_industria', 'ipi_energia', 'ipi_bienes_equipo'] ...
  Columnas export:   ['ipi_corrientes_total_industria', 'ipi_corrientes_energia', 'ipi_corrientes_bienes_equipo'] ...
  Nulos:
    ipi_total_industria: 0
    ipi_energia: 0
    ipi_bienes_equipo: 0
    ipi_bienes_intermedios: 0
    ipi_bienes_consumo: 0

=== IPI constantes base 2025 ===
  Shape: (610, 8)
  Rango: 1975-01-01 → 2025-10-01
  Frecuencia: mensual
  Columnas internas: ['ipi_total_industria', 'ipi_energia', 'ipi_bienes_equipo'] ...
  Columnas export:   ['ipi_constantes_2025_total_industria', 'ipi_constantes_2025_energia', 'ipi_constantes_2025_bienes_equipo'] ...
  Nulos:
    ipi_total_industria: 0
    ipi_energia: 0
    ipi_bienes_equipo: 0
    ipi_bienes_intermedios: 0
    ipi_bienes_consumo: 0



# 10. Exportación de archivos finales

- `ipi_corrientes_1975_2026.csv` - IPI en precios corrientes (IPI × IPRI base 2021 / 100). Columnas con prefijo `ipi_corrientes_*`.
- `ipi_constantes_2025_1975_2026.csv` - IPI en precios constantes base 2025 (corrientes deflactado por IPRI base 2025). Columnas con prefijo `ipi_constantes_2025_*`.


In [43]:
# ── Recortar al rango de tasa de paro y guardar en Datasets ────────────
from pathlib import Path
FECHA_INICIO = '1974-07-01'
FECHA_FIN    = '2025-10-01'
ruta_datasets = Path(r'C:\Users\marco\PycharmProjects\TFM_Marcos\Datasets')

# Añadir columnas temporales trimestre y periodo
for df in [df_corrientes, df_constantes]:
    df['trimestre'] = (df['mes'] - 1) // 3 + 1
    df['periodo']   = df['año'].astype(str) + 'T' + df['trimestre'].astype(str)

# Orden canónico de columnas
time_cols  = ['fecha', 'año', 'mes', 'trimestre', 'periodo']
cols_ipi   = [NOMBRES_IPI[c] for c in CATEGORIAS_IPI]
export_cols = time_cols + cols_ipi

# Exportar corrientes (con prefijo ipi_corrientes_*)
df_corrientes_out = df_corrientes[export_cols].copy()
df_corrientes_out.rename(columns=NOMBRES_CORRIENTES, inplace=True)


df_corrientes_out = df_corrientes_out[(df_corrientes_out['fecha'] >= FECHA_INICIO) & (df_corrientes_out['fecha'] <= FECHA_FIN)].copy()
df_corrientes_out.to_csv(ruta_datasets / 'ipi_corrientes_1975_2026.csv', index=False, encoding='utf-8-sig')
print(f'CORRECTO ipi_corrientes_1975_2026.csv')
print(f'   {df_corrientes_out.shape[0]} meses × {df_corrientes_out.shape[1]} columnas')
print()

# Exportar constantes base 2025 (con prefijo ipi_constantes_2025_*)
df_constantes_out = df_constantes[export_cols].copy()
df_constantes_out.rename(columns=NOMBRES_CONSTANTES, inplace=True)
df_constantes_out = df_constantes_out[(df_constantes_out['fecha'] >= FECHA_INICIO) & (df_constantes_out['fecha'] <= FECHA_FIN)].copy()
df_constantes_out.to_csv(ruta_datasets / 'ipi_constantes_2025_1975_2026.csv', index=False, encoding='utf-8-sig')
print(f'CORRECTO ipi_constantes_2025_1975_2026.csv')
print(f'   {df_constantes_out.shape[0]} meses × {df_constantes_out.shape[1]} columnas')

CORRECTO ipi_corrientes_1975_2026.csv
   610 meses × 10 columnas

CORRECTO ipi_constantes_2025_1975_2026.csv
   610 meses × 10 columnas


# 11. Información acerca de los datasets finales

In [44]:
for nombre, df in [('IPI CORRIENTES', df_corrientes_out), ('IPI CONSTANTES 2025', df_constantes_out)]:
    print(f'Información acerca de {nombre}:')
    print(f'{df.shape[0]} meses × {df.shape[1]} columnas | Rango: {df["fecha"].min().date()} → {df["fecha"].max().date()}')
    columnas = df.columns.tolist()
    print(f'  Columnas: {columnas}')
    nulos = df.isna().sum()
    nulos_col = nulos[nulos > 0]
    if len(nulos_col) > 0:
        print(f'ALERTA  {nombre}: nulos en {nulos_col.to_dict()}')
    else:
        print(f'CORRECTO  {nombre}: sin valores nulos')
    print()
    print(f'=== {nombre} (primeras y últimas 3 filas) ===')
    print(pd.concat([df.head(3), df.tail(3)]).to_string(index=False))
    print()

Información acerca de IPI CORRIENTES:
610 meses × 10 columnas | Rango: 1975-01-01 → 2025-10-01
  Columnas: ['fecha', 'año', 'mes', 'trimestre', 'periodo', 'ipi_corrientes_total_industria', 'ipi_corrientes_energia', 'ipi_corrientes_bienes_equipo', 'ipi_corrientes_bienes_intermedios', 'ipi_corrientes_bienes_consumo']
CORRECTO  IPI CORRIENTES: sin valores nulos

=== IPI CORRIENTES (primeras y últimas 3 filas) ===
     fecha  año  mes  trimestre periodo  ipi_corrientes_total_industria  ipi_corrientes_energia  ipi_corrientes_bienes_equipo  ipi_corrientes_bienes_intermedios  ipi_corrientes_bienes_consumo
1975-01-01 1975    1          1  1975T1                        7.407072                6.449814                     11.800679                          11.266060                      10.792415
1975-02-01 1975    2          1  1975T1                        7.713716                3.225058                     15.583670                           7.752206                       6.222529
1975-03-01

In [45]:
for nombre, df in [('IPI CORRIENTES', df_corrientes_out), ('IPI CONSTANTES 2025', df_constantes_out)]:
    print(f'=== {nombre} ===')
    print(df.describe().round(2))
    print()

=== IPI CORRIENTES ===
                               fecha      año     mes  trimestre  \
count                            610   610.00  610.00     610.00   
mean   2000-05-16 15:30:05.901639296  1999.92    6.48       2.50   
min              1975-01-01 00:00:00  1975.00    1.00       1.00   
25%              1987-09-08 12:00:00  1987.00    3.25       1.25   
50%              2000-05-16 12:00:00  2000.00    6.00       2.00   
75%              2013-01-24 06:00:00  2013.00    9.00       3.00   
max              2025-10-01 00:00:00  2025.00   12.00       4.00   
std                              NaN    14.69    3.45       1.12   

       ipi_corrientes_total_industria  ipi_corrientes_energia  \
count                          610.00                  610.00   
mean                            64.80                   53.47   
min                              5.38                    2.62   
25%                             39.93                   24.85   
50%                             67.95  

In [46]:
for nombre, df in [('IPI CORRIENTES', df_corrientes_out), ('IPI CONSTANTES 2025', df_constantes_out)]:
    print(f'=== {nombre} ===')
    df.info()
    print()

=== IPI CORRIENTES ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 610 entries, 0 to 609
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   fecha                              610 non-null    datetime64[ns]
 1   año                                610 non-null    int64         
 2   mes                                610 non-null    int64         
 3   trimestre                          610 non-null    int64         
 4   periodo                            610 non-null    object        
 5   ipi_corrientes_total_industria     610 non-null    float64       
 6   ipi_corrientes_energia             610 non-null    float64       
 7   ipi_corrientes_bienes_equipo       610 non-null    float64       
 8   ipi_corrientes_bienes_intermedios  610 non-null    float64       
 9   ipi_corrientes_bienes_consumo      610 non-null    float64       
dtypes: datetime64[n